<span style="color: red; font-size: 18px;">Note: The examples in this notebook use TPU v2/v3. Be aware that availability & capacity for these versions can be constrained</span>

# Submit Vertex AI TPU Training from GKE (Prebuilt Container)

This notebook demonstrates submitting a Vertex AI Custom Job with TPU accelerators from a JupyterHub instance running on GKE. Authentication uses Workload Identity — no JSON key files needed.

**What this notebook does:**
1. Verifies Workload Identity authentication
2. Writes a PyTorch/XLA training script (MNIST on TPU)
3. Submits a Vertex AI Custom Job on TPU v5e using a prebuilt container

**Google Cloud services used:** Vertex AI, Cloud Storage, GKE

**Key difference from the custom container approach:** No Dockerfile or Cloud Build step is needed. Vertex AI provides prebuilt PyTorch/XLA containers with TPU support.

## Install Packages

Install the following packages required to execute this notebook.

In [ ]:
! pip3 install --upgrade --quiet google-cloud-aiplatform \
                                 google-cloud-storage

## Verify GKE Environment and Workload Identity

In [ ]:
import subprocess

result = subprocess.run(
    ["cat", "/var/run/secrets/kubernetes.io/serviceaccount/namespace"],
    capture_output=True, text=True,
)
print(f"Running in Kubernetes namespace: {result.stdout}")

from google.auth import default

credentials, project = default()
print(f"Authenticated to project: {project}")
print("Workload Identity is working - no JSON key files needed")

## Set Project Parameters

Set up required parameters.

- `PROJECT_ID`: Google Cloud project ID
- `REGION`: Google Cloud region (e.g. us-central1)

In [ ]:
# Project parameters
PROJECT_ID = "sandbox-401718"          # @param {type:"string"}
REGION = "us-central1"                  # @param {type:"string"}

## TPU Training Configuration

Set up required parameters. Since we are using a prebuilt container, no image build step is needed.

Prebuilt Container:
- `PREBUILT_CONTAINER_URI`: The prebuilt PyTorch/XLA container image from Vertex AI. See [prebuilt containers for training](https://cloud.google.com/vertex-ai/docs/training/pre-built-containers).

Custom Job:
- `SERVICE_ACCOUNT`: The service account to use to run custom jobs
- `PIPELINE_ROOT`: Cloud Storage bucket URI for pipeline artifacts

In [ ]:
# Create the GCS bucket if it doesn't already exist
! gcloud storage buckets create --location {REGION} --project {PROJECT_ID} {PIPELINE_ROOT} 2>/dev/null || echo "Bucket already exists"

In [ ]:
# Prebuilt container for PyTorch/XLA with TPU support
PREBUILT_CONTAINER_URI = "us-docker.pkg.dev/vertex-ai/training/pytorch-xla.2-4.py310:latest"

# Vertex Custom Job parameters
SERVICE_ACCOUNT = "757654702990-compute@developer.gserviceaccount.com"  # @param {type:"string"}
PIPELINE_ROOT = f"gs://{PROJECT_ID}-tpu-pipeline-root"     # @param {type:"string"}

# Working dir for model artifacts
WORKING_DIR = f"{PIPELINE_ROOT}/model"

### Write the Training Script

Write a PyTorch/XLA training script that trains a simple model on MNIST using TPU.

In summary, the training script does the following:

- Gets the directory where to save the model artifacts from the environment variable `AIP_MODEL_DIR`. This variable is set by the training service.
- Loads MNIST dataset from torchvision.
- Builds a simple feed-forward model using PyTorch.
- Discovers and uses TPU devices via `xm.xla_device()`.
- Distributes data loading with `ParallelLoader`.
- Synchronizes optimization steps with `xm.optimizer_step()`.
- Saves the trained model to the specified model directory.

Learn more about [training with TPU accelerators](https://cloud.google.com/vertex-ai/docs/training/training-with-tpu-vm).

In [ ]:
! mkdir -p tpu-training-script

In [ ]:
%%writefile tpu-training-script/train.py
# PyTorch/XLA Training Script for MNIST on TPU

import os
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl
import torch_xla.debug.metrics as met


class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc1 = nn.Linear(784, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 784)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


def train_model(args):
    print("Starting PyTorch XLA training...")
    device = xm.xla_device()
    print(f"Using device: {device}")
    print(f"World size: {xm.xrt_world_size()}")

    model = SimpleModel().to(device)
    optimizer = optim.Adam(model.parameters(), lr=args.lr)
    loss_fn = nn.CrossEntropyLoss()

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    train_dataset = datasets.MNIST(
        '../data', train=True, download=True, transform=transform
    )
    train_sampler = torch.utils.data.distributed.DistributedSampler(
        train_dataset,
        num_replicas=xm.xrt_world_size(),
        rank=xm.get_ordinal(),
        shuffle=True,
    )
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        sampler=train_sampler,
        num_workers=4,
    )

    for epoch in range(1, args.epochs + 1):
        model.train()
        para_train_loader = pl.ParallelLoader(
            train_loader, [device]
        ).per_device_loader(device)
        for batch_idx, (data, target) in enumerate(para_train_loader):
            optimizer.zero_grad()
            output = model(data)
            loss = loss_fn(output, target)
            loss.backward()
            xm.optimizer_step(optimizer)

            if batch_idx % args.log_interval == 0:
                print(
                    f'Train Epoch: {epoch} '
                    f'[{batch_idx * len(data) * xm.xrt_world_size()}/'
                    f'{len(train_loader.dataset)} '
                    f'({100. * batch_idx / len(train_loader):.0f}%)]\t'
                    f'Loss: {loss.item():.6f}'
                )
        print(met.metrics_report())

    print("Training finished.")
    if xm.is_master_ordinal() and args.model_dir:
        torch.save(
            model.state_dict(),
            os.path.join(args.model_dir, "mnist_model.pt"),
        )
        print(f"Model saved to {args.model_dir}")


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument(
        '--model_dir', type=str,
        default=os.environ.get('AIP_MODEL_DIR'),
        help='GCS path for saving the model.',
    )
    parser.add_argument(
        '--batch_size', type=int, default=128,
        help='Input batch size for training.',
    )
    parser.add_argument(
        '--epochs', type=int, default=2,
        help='Number of epochs to train.',
    )
    parser.add_argument(
        '--lr', type=float, default=0.01,
        help='Learning rate.',
    )
    parser.add_argument(
        '--log_interval', type=int, default=10,
        help='How many batches to wait before logging.',
    )
    args = parser.parse_args()
    train_model(args)

## Custom Jobs

Submit TPU training workloads using a [prebuilt container and CustomTrainingJob](https://cloud.google.com/vertex-ai/docs/training/create-custom-job). No Docker build step is required — Vertex AI packages and uploads the script automatically.

In this example, an MNIST classification model is trained using PyTorch/XLA on TPU v5e accelerators.

Learn more about [Training with TPU accelerators](https://cloud.google.com/vertex-ai/docs/training/training-with-tpu-vm).

In [ ]:
from google.cloud import aiplatform

### Set Hardware Accelerators and Training Parameters

Set the TPU machine type for training. For TPU v5e, the machine type bundles CPUs and TPU cores together. Each machine type requires a corresponding `tpu_topology`.

See the [locations where accelerators are available](https://cloud.google.com/vertex-ai/docs/general/locations#accelerators).

In [ ]:
TRAIN_COMPUTE = "ct5lp-hightpu-1t"

TPU_TOPOLOGY_MAP = {
    "ct5lp-hightpu-1t": "1x1",
    "ct5lp-hightpu-4t": "2x2",
    "ct5lp-hightpu-8t": "2x4",
}
TPU_TOPOLOGY = TPU_TOPOLOGY_MAP[TRAIN_COMPUTE]

EPOCHS = 5
BATCH_SIZE = 256

TRAINER_ARGS = [
    "--model_dir", WORKING_DIR,
    "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
]

print("Train machine type:", TRAIN_COMPUTE)
print("TPU Topology:", TPU_TOPOLOGY)
print("Trainer args:", TRAINER_ARGS)

In [ ]:
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=PIPELINE_ROOT)

job = aiplatform.CustomTrainingJob(
    display_name="gke-submitted-tpu-prebuilt-training-job",
    script_path="tpu-training-script/train.py",
    container_uri=PREBUILT_CONTAINER_URI,
)

job.run(
    machine_type=TRAIN_COMPUTE,
    tpu_topology=TPU_TOPOLOGY,
    args=TRAINER_ARGS,
    replica_count=1,
    service_account=SERVICE_ACCOUNT,
    sync=False,
)

In [ ]:
job

## Check Task Completion and Output

Once the job is complete, check to see the output of the TPU training job.

In [ ]:
# Check the status of the custom job
job.state

## Cleanup

To tear down all infrastructure (GKE cluster, IAM, JupyterHub), run `terraform destroy` from the `infra/` directory.

In [ ]:
# Remove the training script artifacts
! rm -rf tpu-training-script

## Additional References

* [Training Code reference from Google Cloud Github](https://github.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/official/pipelines/google_cloud_pipeline_components_TPU_model_train_upload_deploy.ipynb)
* [Training with TPU accelerators](https://cloud.google.com/vertex-ai/docs/training/training-with-tpu-vm)
* [Prebuilt containers for custom training](https://cloud.google.com/vertex-ai/docs/training/pre-built-containers)
* [TPU types](https://cloud.google.com/tpu/docs/system-architecture-tpu-vm)
* [Vertex AI Pipelines](https://cloud.google.com/vertex-ai/docs/pipelines)
* [GKE Workload Identity](https://cloud.google.com/kubernetes-engine/docs/how-to/workload-identity)